In [ ]:
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
from shapely.geometry import Point
import requests

In [ ]:
load_dotenv()

DB_NAME=os.getenv('DB_NAME')
DB_USER=os.getenv('DB_USER')
DB_PW=os.getenv('DB_PW')
DB_HOST=os.getenv('DB_HOST')
DB_PORT=os.getenv('DB_PORT')
SCHEMA='tcc'

In [ ]:
engine = create_engine(f"postgresql://{DB_USER}:{DB_PW}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

## Carga de CISPs (circunscrição de delegacia)

In [ ]:
gdf_cisp = gpd.read_file("../geografia/CISP.shp")[['cisp', 'aisp', 'dp_nome', 'geometry']]
gdf_cisp = gdf_cisp.rename(columns={'dp_nome':'nome_dp'})
gdf_cisp

In [ ]:
cisps_capital = [1,
 4,
 5,
 6,
 7,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44]
gdf_cisp = gdf_cisp[gdf_cisp.cisp.isin(cisps_capital)]
gdf_cisp

In [ ]:
gdf_cisp.plot()

In [ ]:
alvos = ['013ª Ipanema', '012ª Copacabana']
novo_nome = '013ª Ipanema / 012ª Copacabana'

gdf_cisp.loc[gdf_cisp['nome_dp'].isin(alvos), 'nome_dp'] = novo_nome

gdf_cisp = gdf_cisp.dissolve(by='nome_dp', as_index=False, aggfunc='first')

In [ ]:
gdf_cisp

In [ ]:
#gdf = gdf.set_crs("EPSG:4674", allow_override=True)
gdf_cisp.to_postgis("cisp", con=engine, schema=SCHEMA, if_exists='append', index=None)

## Carga dos dados dos bairros
Fonte: data rio

In [ ]:
gpd.read_file("../geografia/bairros.shp")

In [ ]:
gdf_bairros = gpd.read_file("../geografia/bairros.shp")[['nome', 'codbairro', 'Total_de_1', 'geometry', 'Total_de_d']]
gdf_bairros = gdf_bairros.to_crs("EPSG:4326")
gdf_bairros = gdf_bairros.rename(columns={'nome':'nome_bairro', 'codbairro':'id_bairro', 'Total_de_1': 'total_pessoas_2022',
                            'Total_de_d': 'total_domicilios_2022'})
gdf_bairros.plot()

In [ ]:
gdf_cisp.plot()

Colocando a maior parte do bairro dentro de cada cisp

In [ ]:
gdf_cisp = gpd.read_postgis("select * from tcc.cisp", con=engine, geom_col='geometry')

In [ ]:
# 1) Garantir mesmo CRS para ambos (geográfico)
gdf_bairros = gdf_bairros.to_crs(gdf_cisp.crs)

# 2) Reprojetar para CRS em metros (UTM Rio de Janeiro)
bairros_m = gdf_bairros.to_crs("EPSG:31983")
cisps_m   = gdf_cisp.to_crs("EPSG:31983")

# 3) Interseção
inter = gpd.overlay(
    bairros_m,
    cisps_m[['id', 'geometry']],
    how='intersection'
)

# 4) Calcular área da interseção corretamente
inter["area_inter"] = inter.area

# 5) Selecionar CISP que cobre maior parte do bairro
inter_sorted = inter.sort_values("area_inter", ascending=False)
bairros_cisp = inter_sorted.drop_duplicates(subset="nome_bairro")

# 6) Trazer o id da CISP de volta para o CRS original
bairros_cisp = bairros_cisp[['nome_bairro', 'id']]

gdf_bairros_final = gdf_bairros.merge(
    bairros_cisp, 
    on="nome_bairro", 
    how="left"
).rename(columns={'id':'cisp_id'})


In [ ]:
gdf_bairros_final.head()

In [ ]:
gdf_bairros_zona = gpd.read_file('../geografia/Limite_de_Bairros.shp')[['codbairro', 'area_plane']]
gdf_bairros_zona.sort_values('codbairro')

In [ ]:
gdf_bairros_final.merge(gdf_bairros_zona, how='left', left_on='id_bairro', right_on='codbairro').sort_values('id_bairro')

In [ ]:
gdf = gdf_bairros_final.merge(gdf_bairros_zona, how='inner', left_on='id_bairro', right_on='codbairro')
mapeamento_zona = {
    '1': "Zona Norte",
    '2': "Zona Sul",
    '3': "Zona Norte",
    '5': "Zona Oeste",
    '4': "Zona Sudoeste"
}

gdf['id_bairro'] = gdf['id_bairro'].astype(str).str.lstrip('0').astype(int)


# Cria a nova coluna 'zona' com base no número em texto
gdf['zona'] = gdf['area_plane'].astype(str).map(mapeamento_zona)

# Corrige os bairros 32–38 para "Zona Norte"
gdf.loc[gdf['id_bairro'].between(32, 38), 'zona'] = "Zona Norte"

gdf.loc[gdf['id_bairro'].isin([1, 2, 3, 8, 9, 6, 5]), 'zona'] = "Centro"

gdf = gdf.drop(columns=['area_plane', 'codbairro'])
gdf

In [ ]:
import matplotlib.pyplot as plt
gdf.plot(column='zona',
         cmap='tab20',
         legend=True,
         figsize=(10, 8),
         edgecolor='black',
         linewidth=0.3)

plt.title("Mapa por Zona", fontsize=14)
plt.show()

In [ ]:
gdf

In [ ]:
gdf.to_postgis("bairro", con=engine, schema=SCHEMA, if_exists='append')

## Dados do fogo cruzado

In [ ]:
from main_api_fogo_cruzado_automatico import main as fogo_cruzado
fogo_cruzado()

In [ ]:
df_fogo_cruzado = pd.read_pickle("../fogo_cruzado/resultado_fogo_cruzado.pkl")
df_fogo_cruzado = df_fogo_cruzado[df_fogo_cruzado.data_ocorrencia.dt.year==2025]
gdf_fogo_cruzado = gpd.GeoDataFrame(df_fogo_cruzado, geometry='geometry')
gdf_fogo_cruzado

In [ ]:
mapeamento = {
    'id_ocorrencia': 'id',
    'local_ocorrencia': 'endereco',
    'geometry': 'geom',
    'data_ocorrencia': 'data_ocorrencia',
    'presen_agen_segur_ocorrencia': 'agente_presente',
    'chacina_unidades_policiais_oc': 'unidade_poliicia',
    'motivo_principal': 'motivo_principal',
    'motivo_complementar': 'motivo_secundario',
    'chacina_oc': 'masssacre',
    'qtd_morto_agen_segur_ocorrencia': 'qtd_agentes_mortos',
    'qtd_ferido_agen_segur_ocorrencia': 'qtd_agentes_feridos',
    'homem_qtd_mortos_oc': 'qtd_homens_mortos',
    'homem_qtd_feridos_oc': 'qtd_homens_feridos',
    'mulher_qtd_mortos_oc': 'qtd_mulheres_mortas',
    'mulher_qtd_feridos_oc': 'qtd_mulheres_feridas',
    'vitima_crianca_qtd_mortos_oc': 'qtd_criancas_mortas',
    'vitima_crianca_qtd_feridos_oc': 'qtd_criancas_feridas',
    'vitima_adolescente_qtd_mortos_oc': 'qtd_adolescentes_mortos',
    'vitima_adolescente_qtd_feridos_oc': 'qtd_adolescentes_feridos',
    'vitima_idoso_qtd_mortos_oc': 'qtd_idosos_mortos',
    'vitima_idoso_qtd_feridos_oc': 'qtd_idosos_feridos',
    'qtd_morto_civil_ocorrencia': 'qtd_civis_mortos',
    'qtd_ferido_civil_ocorrencia': 'qtd_civis_feridos'
}
gdf_fogo_cruzado_2 = gdf_fogo_cruzado.rename(columns=mapeamento)

# Campos fixos ausentes na API
gdf_fogo_cruzado_2['subbairro'] = None
gdf_fogo_cruzado_2['localidade'] = None  # como solicitado

# Campo derivado
gdf_fogo_cruzado_2['acao_da_policia'] = gdf_fogo_cruzado_2['agente_presente']


In [ ]:
gdf_bairros = gpd.read_postgis("select id_bairro, geometry from bairro", con=engine, geom_col='geometry')
gdf_bairros

In [ ]:
gdf_fc = gdf_fogo_cruzado_2.set_geometry("geom")
gdf_fc = gdf_fc.set_crs(4326, allow_override=True)
gdf_fc = gdf_fc.to_crs(4326)

gdf_fc = gdf_fc.sjoin(
    gdf_bairros,
    how='left',
    predicate='within'
).rename(columns={'id_bairro':'bairro_id_bairro'})

gdf_fc

In [ ]:
gdf_fc = gdf_fc[(gdf_fc.nome_cidade == 'Rio de Janeiro') & (gdf_fc.data_ocorrencia.dt.year == 2025)]
gdf_fc

In [ ]:
cols_banco = [
    'bairro_id_bairro', 'id', 'endereco', 'subbairro', 'localidade', 'geom',
    'data_ocorrencia', 'acao_da_policia', 'agente_presente',
    'unidade_poliicia', 'motivo_principal', 'motivo_secundario','qtd_agentes_mortos', 'qtd_agentes_feridos',
    'qtd_homens_mortos', 'qtd_homens_feridos', 'qtd_mulheres_mortas',
    'qtd_mulheres_feridas', 'qtd_criancas_mortas', 'qtd_criancas_feridas',
    'qtd_adolescentes_mortos', 'qtd_adolescentes_feridos',
    'qtd_idosos_mortos', 'qtd_idosos_feridos', 'qtd_civis_mortos',
    'qtd_civis_feridos'
]
gdf_fc = gdf_fc[cols_banco]
gdf_fc['bairro_id_bairro'] = gdf_fc['bairro_id_bairro'].astype('Int64')

In [ ]:
gdf_fc = gdf_fc[~pd.isna(gdf_fc.bairro_id_bairro)]

In [ ]:
bool_map = {
    'Sim': True, 'Não': False,
    'Nao': False, 'sim': True,
    'nao': False, None: None
}

for col in ['acao_da_policia', 'agente_presente']:
    if col in gdf_fc.columns:
        gdf_fc[col] = gdf_fc[col].map(bool_map).astype('boolean')

gdf_fc.to_postgis("ocorrencia_tiroteio", con=engine, schema=SCHEMA, if_exists='append', index=False)

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    # Inserir na tabela tcc.cisp
    conn.execute(
        text("""
        INSERT INTO tcc.cisp (id, cisp, aisp, nome_dp, geometry)
        VALUES (:id, :cisp, :aisp, :nome_dp, :geometry)
        """),
        {"id": 0, "cisp": 0, "aisp": 0, "nome_dp": "Sem DP", "geometry": None}
    )

    # Inserir na tabela tcc.crime
    conn.execute(
        text("""
        INSERT INTO tcc.bairro (
            id_bairro, nome_bairro, geometry, cisp_id, total_pessoas_2022, total_domicilios_2022, zona
        ) VALUES (
            :id_bairro, :nome_bairro, :geometry, :cisp_id, :total_pessoas_2022, :total_domicilios_2022, :zona
        )
        """),
        {
            "id_bairro": 0,
            "nome_bairro": "Sem Bairro",
            "geometry": None,
            "cisp_id": None,
            "total_pessoas_2022": None,
            "total_domicilios_2022": None,
            "zona": None
        }
    )

    # Commit automático se estiver em autocommit False
    conn.commit()
